<a href="https://colab.research.google.com/github/bored-shinigami2805/CVPR-SLM-fine-tuning/blob/main/chatLoop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chat UI - fine-tuned CV assistant (Qwen2.5-3B-Instruct + LoRA)

Uses **Gradio** for a reliable text box + chat transcript with live streaming.
(ipywidgets + threaded streaming hangs in Colab; Gradio does not.)

**Runtime -> Change runtime type -> GPU.** Run the 3 cells top to bottom. The last cell
prints a link and shows the chat inline - type in the box and press Enter.

> If a previous cell is stuck on a spinner: **Runtime -> Interrupt execution** first.

## 1. Install

In [1]:
!pip -q install -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" bitsandbytes gradio
import torch; print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 which is incompatible.
cuda available: True


## 2. Load model + adapter

Set `ADAPTER_DIR` to your saved adapter, or `None` to chat with the base model.
(`/content/...` is wiped between Colab sessions; on a fresh runtime, re-train first or
load the adapter from Google Drive.)

In [2]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_ID     = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "/content/qwen2p5-3b-cv-lora"   # or None for the base model
USE_4BIT    = True

SYS = ("You are a computer-vision research assistant. You have studied a specific set of "
       "computer-vision papers in depth. Answer accurately and concisely, grounded only in "
       "what you know. If a question is outside computer vision, say it is outside your domain. "
       "If it is about computer vision but not something you have studied, say you don't have "
       "that information rather than guessing.")

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR if ADAPTER_DIR and os.path.isdir(ADAPTER_DIR) else BASE_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

quant = None
if USE_4BIT:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

model = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=quant, torch_dtype=torch.bfloat16, device_map="auto")
if ADAPTER_DIR and os.path.isdir(ADAPTER_DIR):
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    print("loaded LoRA adapter from", ADAPTER_DIR)
else:
    print("running BASE model (no adapter)")
model.eval(); model.config.use_cache = True

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

running BASE model (no adapter)


## 3. Launch the chat

A text box with streaming replies. Press Enter to send. Use the **Clear** button to reset.
The generator is greedy (deterministic) - good for checking the grounded / refuse / defer
behavior. A public share link is also printed.

In [4]:
import threading
import gradio as gr
from transformers import TextIteratorStreamer

def respond(message, history):
    # Build the full conversation, robust to either gradio history format.
    msgs = [{"role": "system", "content": SYS}]
    for h in (history or []):
        if isinstance(h, dict) and h.get("content"):
            msgs.append({"role": h.get("role", "user"), "content": h["content"]})
        elif isinstance(h, (list, tuple)) and len(h) == 2:
            u, a = h
            if u: msgs.append({"role": "user", "content": u})
            if a: msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})

    enc = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    streamer = TextIteratorStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    kw = dict(**enc, max_new_tokens=512, do_sample=False, streamer=streamer,
              pad_token_id=tok.pad_token_id, repetition_penalty=1.05)

    err = {}
    def _run():
        try:
            with torch.no_grad():
                model.generate(**kw)
        except Exception as e:          # never let the streamer hang the UI
            err["e"] = e
            streamer.end()
    threading.Thread(target=_run, daemon=True).start()

    partial = ""
    for piece in streamer:
        partial += piece
        yield partial
    if err:
        yield partial + f"\n\n[generation error: {err['e']}]"

demo = gr.ChatInterface(
    fn=respond,
    title="CV research assistant (Qwen2.5-3B + LoRA)",
    description="Ask about the studied computer-vision papers.")
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8646f543d26ab878c5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
